In [0]:
# ============================================================
# BRONZE LAYER — Raw CSV Ingestion into Delta Lake
# ============================================================

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.bronze")

DataFrame[]

In [0]:
spark.sql("SHOW VOLUMES IN workspace.default").show()

+--------+-----------+
|database|volume_name|
+--------+-----------+
| default|     bronze|
| default|   raw_data|
+--------+-----------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

spark = SparkSession.builder.getOrCreate()

In [0]:
# ── Config ───────────────────────────────────────────────────
RAW_PATH   = "/Volumes/workspace/default/raw_data/Brazil E-Commerce/"

In [0]:
# ── Table registry ───────────────────────────────────────────
TABLES = {
    "orders"        : ("olist_orders_dataset.csv",               "order_id"),
    "order_items"   : ("olist_order_items_dataset.csv",          "order_id"),
    "payments"      : ("olist_order_payments_dataset.csv",       "order_id"),
    "reviews"       : ("olist_order_reviews_dataset.csv",        "review_id"),
    "customers"     : ("olist_customers_dataset.csv",            "customer_id"),
    "sellers"       : ("olist_sellers_dataset.csv",              "seller_id"),
    "products"      : ("olist_products_dataset.csv",             "product_id"),
    "geolocation"   : ("olist_geolocation_dataset.csv",          "geolocation_zip_code_prefix"),
    "category_names": ("product_category_name_translation.csv",  "product_category_name"),
}

In [0]:
# ── Ingestion function ───────────────────────────────────────
def ingest_to_bronze(table_name, csv_file, pk_col):
    print(f"\n>>> Ingesting: {table_name}")

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .csv(f"{RAW_PATH}{csv_file}")
    )

    df = (
        df.withColumn("_ingested_at", current_timestamp())
          .withColumn("_source_file", lit(csv_file))
    )

    row_count  = df.count()
    null_count = df.filter(df[pk_col].isNull()).count()
    dup_count  = row_count - df.dropDuplicates([pk_col]).count()

    print(f"    Rows       : {row_count:,}")
    print(f"    Null PKs   : {null_count}")
    print(f"    Duplicates : {dup_count}")

    # Write as managed Delta table — no path required
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"ecommerce_bronze.{table_name}")
        #Databricks handles storage location internally.
    )

    print(f"    Saved to   : ecommerce_bronze.{table_name}")
    return {"table": table_name, "rows": row_count, "null_pks": null_count, "duplicates": dup_count}


In [0]:
# ── Run all tables ───────────────────────────────────────────
results = []
for table_name, (csv_file, pk_col) in TABLES.items():
    result = ingest_to_bronze(table_name, csv_file, pk_col)
    results.append(result)


>>> Ingesting: orders
    Rows       : 99,441
    Null PKs   : 0
    Duplicates : 0
    Saved to   : ecommerce_bronze.orders

>>> Ingesting: order_items
    Rows       : 112,650
    Null PKs   : 0
    Duplicates : 13984
    Saved to   : ecommerce_bronze.order_items

>>> Ingesting: payments
    Rows       : 103,886
    Null PKs   : 0
    Duplicates : 4446
    Saved to   : ecommerce_bronze.payments

>>> Ingesting: reviews
    Rows       : 99,224
    Null PKs   : 0
    Duplicates : 814
    Saved to   : ecommerce_bronze.reviews

>>> Ingesting: customers
    Rows       : 99,441
    Null PKs   : 0
    Duplicates : 0
    Saved to   : ecommerce_bronze.customers

>>> Ingesting: sellers
    Rows       : 3,095
    Null PKs   : 0
    Duplicates : 0
    Saved to   : ecommerce_bronze.sellers

>>> Ingesting: products
    Rows       : 32,951
    Null PKs   : 0
    Duplicates : 0
    Saved to   : ecommerce_bronze.products

>>> Ingesting: geolocation
    Rows       : 1,000,163
    Null PKs   : 0
    Du

In [0]:
# ── Summary ──────────────────────────────────────────────────
print("\n" + "="*55)
print("BRONZE INGESTION SUMMARY")
print("="*55)
for r in results:
    status = "✅" if r["null_pks"] == 0 and r["duplicates"] == 0 else "⚠️"
    print(f"{status}  {r['table']:<20} | rows: {r['rows']:>7,} | null_pks: {r['null_pks']} | dups: {r['duplicates']}")


BRONZE INGESTION SUMMARY
✅  orders               | rows:  99,441 | null_pks: 0 | dups: 0
⚠️  order_items          | rows: 112,650 | null_pks: 0 | dups: 13984
⚠️  payments             | rows: 103,886 | null_pks: 0 | dups: 4446
⚠️  reviews              | rows:  99,224 | null_pks: 0 | dups: 814
✅  customers            | rows:  99,441 | null_pks: 0 | dups: 0
✅  sellers              | rows:   3,095 | null_pks: 0 | dups: 0
✅  products             | rows:  32,951 | null_pks: 0 | dups: 0
⚠️  geolocation          | rows: 1,000,163 | null_pks: 0 | dups: 981148
✅  category_names       | rows:      71 | null_pks: 0 | dups: 0
